## Observação: 
 
#### Com os 54.840 pontos de dado este notebook demora em média 40 minutos para rodar na minha máquina.

In [84]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import json
import glob

#gensim
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

#Scikit Learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearnlearn

#spacy
import spacy

#vis
import pyLDAvis
import pyLDAvis.gensim_models

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [72]:
df = pd.read_csv('database/journals.csv')
df = df.iloc[np.where(~df['Abstract'].isna())]
df = df.iloc[np.where(~df['Publication Year'].isna())]

# Descritivas

In [73]:
print(len(df))

53038


In [74]:
df['Source Title'].value_counts().sort_index()

ARTIFICIAL INTELLIGENCE REVIEW                                     1378
COMPUTATIONAL LINGUISTICS                                           574
EXPERT SYSTEMS WITH APPLICATIONS                                  15809
FOUNDATIONS AND TRENDS IN MACHINE LEARNING                           22
IEEE COMPUTATIONAL INTELLIGENCE MAGAZINE                            267
IEEE TRANSACTIONS ON KNOWLEDGE AND DATA ENGINEERING                4313
IEEE TRANSACTIONS ON NEURAL NETWORKS AND LEARNING SYSTEMS          3292
IEEE TRANSACTIONS ON PATTERN ANALYSIS AND MACHINE INTELLIGENCE     5609
IEEE Transactions on Neural Networks and Learning Systems             5
INTERNATIONAL JOURNAL OF COMPUTER VISION                           2365
INTERNATIONAL JOURNAL OF INTELLIGENT SYSTEMS                       2318
INTERNATIONAL JOURNAL OF ROBOTICS RESEARCH                         2195
JOURNAL OF ARTIFICIAL INTELLIGENCE RESEARCH                        1363
JOURNAL OF MACHINE LEARNING RESEARCH                            

In [75]:
df['Publication Year'].value_counts().sort_index()

1990.0      10
1991.0     406
1992.0     529
1993.0     600
1994.0     600
1995.0     683
1996.0     716
1997.0     719
1998.0     760
1999.0     713
2000.0     684
2001.0     738
2002.0     877
2003.0     935
2004.0    1089
2005.0    1101
2006.0    1178
2007.0    1305
2008.0    1663
2009.0    2467
2010.0    2258
2011.0    2819
2012.0    2883
2013.0    2192
2014.0    2259
2015.0    2391
2016.0    2293
2017.0    2305
2018.0    2695
2019.0    2668
2020.0    3093
2021.0    4302
2022.0    3107
Name: Publication Year, dtype: int64

### Pontos de dado com a data disponível variando mensalmente

In [76]:
sum(df['Publication Date'].value_counts()[:12])

37396

### Pontos de dado com a data disponível variando diariamente

In [77]:
sum(df['Publication Date'].value_counts()[12:])

12116

In [78]:
data = list(df['Abstract'])

In [79]:
def lemmatization(texts, allowed_postags=["NOUN", "ADJ", "VERB", "ADV"]):
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    return [" ".join([token.lemma_ for token in nlp(text) if token.pos_ in allowed_postags]) for text in texts]

lemmatized_texts = lemmatization(data)

In [80]:
def gen_words(texts):
    return [gensim.utils.simple_preprocess(text, deacc=True) for text in texts]

data_words = gen_words(lemmatized_texts)

In [85]:
#BIGRAMS
bigram_phrases = gensim.models.Phrases(data_words, min_count=5, threshold=50)
trigram_phrases = gensim.models.Phrases(bigram_phrases[data_words], threshold=50)

bigram = gensim.models.phrases.Phraser(bigram_phrases)
trigram = gensim.models.phrases.Phraser(trigram_phrases)

def make_bigrams(texts):
    return [bigram[doc] for doc in texts]

def make_trigrams(texts):
    return [trigram[bigram[doc]] for doc in texts]

data_bigrams = make_bigrams(data_words)
data_bigrams_trigrams = make_trigrams(data_bigrams) 

In [100]:
import itertools

id2word = corpora.Dictionary(data_bigrams)
texts = data_bigrams
corpus = [id2word.doc2bow(text) for text in texts]

In [155]:
#TF-IDF
id2word = corpora.Dictionary(data_bigrams)
texts = data_bigrams
corpus = [id2word.doc2bow(text) for text in texts]

tfidf = TfidfModel(corpus, id2word=id2word)

low_value=0.2
words = []
words_missing_in_tfidf = []

for i in range(0, len(corpus)):
    bow = corpus[i]
    tfidf_ids = [id for id,value in tfidf[bow]]
    bow_ids = [id for id, value in bow]
    low_value_words = [id for id in bow_ids if tfidf.dfs[id]/len(corpus) > low_value]
    drops = low_value_words + words_missing_in_tfidf
    for item in drops:
        words.append(id2word[item])
    words_missing_in_tfidf = [id for id in bow_ids if id not in tfidf_ids] # The words with tf-idf socre 0 will be missing

    new_bow = [b for b in bow if b[0] not in low_value_words and b[0] not in words_missing_in_tfidf]       
    corpus[i] = new_bow

In [156]:
print('percentual de palavras dropadas = ',len(words)/len(c_words))
print('numero de palavras dropadas = ',len(words))

percentual de palavras dropadas =  0.16810775246672566
numero de palavras dropadas =  626760


# MODEL MULTICORE

In [157]:
lda_model = gensim.models.ldamulticore.LdaMulticore(corpus=corpus,
                                           id2word=id2word,
                                           num_topics=30,
                                           random_state=100,
                                           chunksize=100,
                                           passes=10)

# MODEL SINGLE CORE

In [ ]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                           id2word=id2word,
                                           num_topics=30,
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=10,
                                           alpha="auto")

# Visualization

In [158]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_model, corpus, id2word, mds="mmds", R=30)
vis

PreparedData(topic_coordinates=              x         y  topics  cluster      Freq
topic                                               
25    -0.060877 -0.024572       1        1  7.111843
24     0.093499 -0.084356       2        1  6.591769
9      0.068800  0.137988       3        1  6.364353
2     -0.104494  0.111766       4        1  5.683678
4      0.213316 -0.034310       5        1  5.296585
12     0.122308  0.087292       6        1  5.059461
13     0.039784  0.256207       7        1  4.434337
21    -0.075142 -0.170555       8        1  4.089605
17    -0.235855  0.191572       9        1  4.070278
16     0.110409  0.341481      10        1  3.916205
20     0.227395  0.102972      11        1  3.869482
19    -0.037060  0.364367      12        1  3.731896
1     -0.194924 -0.044906      13        1  3.529909
18    -0.126881  0.289336      14        1  3.510073
27     0.021280 -0.266015      15        1  3.287081
23    -0.234547  0.341926      16        1  3.108331
15    -0.342214 -0.044455      17        1  2.989364
11    -0.201892  0.093196      18        1  2.860378
7      0.358056  0.055811      19        1  2.827806
6      0.305563  0.230220      20        1  2.641264
3     -0.396626  0.123892      21        1  2.009845
28     0.191670 -0.208823      22        1  1.980845
29     0.227736  0.321512      23        1  1.924858
8     -0.193591 -0.356007      24        1  1.884822
5     -0.264031 -0.240075      25        1  1.704718
14     0.358634 -0.282411      26        1  1.458193
26     0.436654 -0.117590      27        1  1.226901
10    -0.443157 -0.204106      28        1  1.175334
0      0.199964 -0.463083      29        1  0.971699
22    -0.063779 -0.508275      30        1  0.689089, topic_info=                Term          Freq         Total Category  logprob  loglift
1129           class  12495.000000  12495.000000  Default  30.0000  30.0000
83    classification  17236.000000  17236.000000  Default  29.0000  29.0000
518           object  17436.000000  17436.000000  Default  28.0000  28.0000
1086            user  11344.000000  11344.000000  Default  27.0000  27.0000
31         knowledge  13341.000000  13341.000000  Default  26.0000  26.0000
...              ...           ...           ...      ...      ...      ...
742       biological    398.928710    746.616280  Topic30  -4.3026   4.3508
2690         subject    748.829689   2100.329876  Topic30  -3.6729   3.9462
940            right    374.732315    774.756636  Topic30  -4.3652   4.2512
1048       coherence    218.413324    301.021263  Topic30  -4.9050   4.6568
3048   factorization    207.193168    367.263193  Topic30  -4.9577   4.4051

[2016 rows x 6 columns], token_table=      Topic      Freq         Term
term                              
1935      7  0.993502     ablation
1530     21  0.997350     abnormal
2201     21  0.994655  abnormality
7562      6  0.986190       absent
5111      7  0.108675  abstraction
...     ...       ...          ...
325      13  0.007871         year
325      14  0.034981         year
325      20  0.198515         year
325      21  0.062090         year
3264      1  0.993977         zone

[7963 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[26, 25, 10, 3, 5, 13, 14, 22, 18, 17, 21, 20, 2, 19, 28, 24, 16, 12, 8, 7, 4, 29, 30, 9, 6, 15, 27, 11, 1, 23])